<a href="https://colab.research.google.com/github/nikhilmooloo02-stack/CLARIX_AI_AGENT/blob/main/CLARIX_AI_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# CELL 1 - Install dependencies
!pip install anthropic gradio chromadb pypdf2 pillow requests -q
print("All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [4]:
# CELL 2 - Connect to Claude AI
import anthropic
from google.colab import userdata

# Get API key securely
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Test connection
def test_connection():
    print(client.messages.create(
      model="claude-sonnet-4-5",
      max_tokens=100,
      messages=[{"role": "user", "content": "Say: HI MY NAME IS CLARIX, YOUR AI ASSISTANT. HOW CAN I HELP YOU TODAY?."}]
    ).content[0].text)

test_connection()


HI MY NAME IS CLARIX, YOUR AI ASSISTANT. HOW CAN I HELP YOU TODAY?


In [7]:
SYSTEM_PROMPT = """You are CLARIX, a professional AI customer support agent
for South African SMEs. Your job is to:
- Handle customer complaints with empathy
- Provide clear solutions and timelines
- Escalate serious issues when necessary
- Always acknowledge the customer's frustration first
- Be concise and professional"""

conversation_history = []

def ask_clarix(user_message):
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=conversation_history
    )

    reply = response.content[0].text
    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    print(f"\nCLARIX: {reply}\n")

# Test it
ask_clarix("A customer says their order arrived damaged and wants a refund.")


CLARIX: I understand how frustrating it must be to receive a damaged order. I'm sorry this happened, and I'll help you resolve this right away.

**Here's what I can do for you:**

1. **Immediate refund** - I can process a full refund to your original payment method within 3-5 business days

2. **Replacement option** - Or, if you'd prefer, I can send a replacement item with expedited shipping at no charge

**What I need from you:**
- A quick photo of the damaged item (helps us improve our packaging)
- Your order number

**Next steps:**
- Once you provide the photo, I'll process your chosen solution immediately
- You'll receive a confirmation email within the hour
- For the damaged item, you can dispose of it - no need to return it

Which option works better for you - refund or replacement?

Is there anything else about this order I can help clarify?



In [9]:
# Cell 4 — CLARIX Gradio UI
import gradio as gr

def chat(message, history):
    conversation = []
    for human, assistant in history:
        conversation.append({"role": "user", "content": human})
        conversation.append({"role": "assistant", "content": assistant})

    conversation.append({"role": "user", "content": message})

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=conversation
    )

    return response.content[0].text

demo = gr.ChatInterface(
    fn=chat,
    title="CLARIX — AI Customer Support Agent",
    description="Professional AI-powered customer support for your business.",
    examples=[
        "My order arrived damaged and I want a refund.",
        "I've been waiting 3 weeks and nobody is responding.",
        "I want to cancel my order immediately.",
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f1143e5bbe38e84b6a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
!git clone https://github.com/nikhilmooloo02-stack/CLARIX_AI_AGENT.git

Cloning into 'CLARIX_AI_AGENT'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.
